# Visual Question Answering with ResNet18 and Bi-LSTM

A binary VQA baseline: combine an image with a question and predict **yes** or **no**.

Course project from AI VIET NAM, AIO2026, Module 5.

[Repository](https://github.com/ducnguyen1110/vqa-cnn-lstm-5) ·
[Open in Colab](https://colab.research.google.com/github/ducnguyen1110/vqa-cnn-lstm-5/blob/main/notebooks/vqa_cnn_lstm.ipynb)

Run cells in order. On Colab, select a GPU runtime before training. Locally,
launch Jupyter from the repository root. The dataset and checkpoints are written
to `data/` and `checkpoints/` in the working directory.

This cleaned notebook has not been retrained. The supplied notebook's recorded
16-epoch run is documented separately in `reports/experiment.md`.


## 0. Environment

Install the notebook dependencies. PyTorch and torchvision must be a compatible
pair; Colab normally provides both. The vocabulary is built with Python rather
than the retired torchtext package. A blank English spaCy pipeline provides the
tokenizer, so no external language-model download is needed.


In [ ]:
%pip install -q "torch>=2.2,<3" "torchvision>=0.17,<1" "timm>=1,<2" "spacy>=3.7,<4" "gdown>=5,<6" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "Pillow>=10,<13"


## 1. Dataset

Download the course-provided COCO-derived yes/no subset. The archive must contain
`vaq2.0.TrainImages.txt`, `vaq2.0.DevImages.txt`, `vaq2.0.TestImages.txt`, and
`val2014-resised/` (the original spelling is intentional).

If Google Drive limits downloads, obtain the archive from the course dataset
folder and place it at `data/vqa_coco_dataset.zip`, then rerun this cell.
Dataset access and redistribution rights remain with the original providers.


In [ ]:
from pathlib import Path
import zipfile
import gdown

DATA_DIR = Path('data')
CHECKPOINT_DIR = Path('checkpoints')
DATA_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)
DATASET_FILE_ID = '1_Im0Yl2yAL5sHXQdBfPoW6pFQMANYySJ'
archive = DATA_DIR / 'vqa_coco_dataset.zip'
expected = ['vaq2.0.TrainImages.txt', 'vaq2.0.DevImages.txt',
            'vaq2.0.TestImages.txt', 'val2014-resised']
if not all((DATA_DIR / name).exists() for name in expected):
    if not archive.exists():
        downloaded = gdown.download(id=DATASET_FILE_ID, output=str(archive), quiet=False)
        if downloaded is None:
            raise RuntimeError('Download failed. Place the course archive in data/ and rerun.')
    with zipfile.ZipFile(archive) as dataset_zip:
        target = DATA_DIR.resolve()
        for member in dataset_zip.infolist():
            if not (target / member.filename).resolve().is_relative_to(target):
                raise ValueError('Archive contains an unsafe path.')
        dataset_zip.extractall(target)
missing = [name for name in expected if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Expected files directly inside {DATA_DIR}: {missing}')
IMAGE_DIR = DATA_DIR / 'val2014-resised'


## 2. Import libraries and set random seed

In [ ]:
import json
import os
import random
import platform
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import spacy
import timm
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print({'python': platform.python_version(), 'torch': torch.__version__,
       'torchvision': torchvision.__version__, 'timm': timm.__version__,
       'spacy': spacy.__version__, 'numpy': np.__version__})


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 59
set_seed(seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


## 3. Read dataset

In [ ]:
def load_data(set_path):
    data = []
    with open(set_path, encoding='utf-8') as file:
        for line_no, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                image_field, qa_field = line.rstrip().split('\t', 1)
                question, answer = qa_field.rsplit('?', 1)
                answer = answer.strip().lower()
                # Course annotations append a '#<digit>' suffix to filenames.
                image_path = image_field.rsplit('#', 1)[0]
                if Path(image_path).name != image_path:
                    raise ValueError('Expected an image filename, without directories')
                if answer not in {'yes', 'no'}:
                    raise ValueError(f'Unexpected label: {answer}')
                data.append({'image_path': image_path,
                             'question': question.split('?')[0].strip() + '?',
                             'answer': answer})
            except ValueError as error:
                raise ValueError(f'{set_path}:{line_no}: {error}') from error
    if not data:
        raise ValueError(f'No samples found in {set_path}')
    return data

train_data = load_data(DATA_DIR / 'vaq2.0.TrainImages.txt')
val_data = load_data(DATA_DIR / 'vaq2.0.DevImages.txt')
test_data = load_data(DATA_DIR / 'vaq2.0.TestImages.txt')
print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print(train_data[0])


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, sample in zip(axes, train_data[:4]):
    img = Image.open(os.path.join(IMAGE_DIR, sample['image_path']))
    ax.imshow(img)
    ax.set_title(f"{sample['question']}\nAnswer: {sample['answer']}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
splits = {'Train': train_data, 'Val': val_data, 'Test': test_data}

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# Share of 'yes' / 'no' answers in each split
names = list(splits)
yes_ratio = [np.mean([s['answer'] == 'yes' for s in d]) for d in splits.values()]
no_ratio = [1 - r for r in yes_ratio]
ax[0].bar(names, no_ratio, label='no', color='tab:red')
ax[0].bar(names, yes_ratio, bottom=no_ratio, label='yes', color='tab:green')
for i, (n, y) in enumerate(zip(no_ratio, yes_ratio)):
    ax[0].text(i, n / 2, f'{n:.1%}', ha='center', color='white')
    ax[0].text(i, n + y / 2, f'{y:.1%}', ha='center', color='white')
ax[0].set_title('Answer distribution per split')
ax[0].set_ylabel('Ratio')
ax[0].set_ylim(0, 1.15)
ax[0].legend(ncol=2, loc='upper center')

# Question length in words (the '?' at the end is not counted)
lengths = [len(s['question'].replace('?', '').split()) for s in train_data]
ax[1].hist(lengths, bins=np.arange(min(lengths), max(lengths) + 2) - 0.5, color='tab:blue', edgecolor='white')
ax[1].set_title('Question length (train)')
ax[1].set_xlabel('Number of words')
ax[1].set_ylabel('Number of questions')
plt.tight_layout()
plt.show()

for name, d in splits.items():
    n_yes = sum(s['answer'] == 'yes' for s in d)
    print(f'{name}: {len(d)} samples ({n_yes} yes, {len(d) - n_yes} no)')


In [ ]:
# sorted() keeps the class order fixed across runs (a plain set has no fixed order)
classes = sorted(set([sample['answer'] for sample in train_data]))
label2idx = {
    cls_name: idx for idx, cls_name in enumerate(classes)
}
idx2label = {
    idx: cls_name for idx, cls_name in enumerate(classes)
}
print(idx2label)


## 4. Data preprocessing

In [ ]:
eng = spacy.blank('en')
counter = Counter(token.text for sample in train_data
                  for token in eng.tokenizer(sample['question']))
special_tokens = ['<pad>', '<sos>', '<eos>', '<unk>']
# Match the legacy vocabulary ordering: frequency first, alphabetical on ties.
words = sorted(counter, key=lambda token: (-counter[token], token))
vocab = {token: index for index, token in enumerate(
    special_tokens + [token for token in words
                      if counter[token] >= 2 and token not in special_tokens])}
print('Vocabulary size:', len(vocab))


In [ ]:
def tokenize(question, max_seq_len):
    tokens = [token.text for token in eng.tokenizer(question)]
    sequence = [vocab.get(token, vocab['<unk>']) for token in tokens][:max_seq_len]
    return sequence + [vocab['<pad>']] * (max_seq_len - len(sequence))

print(tokenize('Is this a picture of an apple?', max_seq_len=20))


In [ ]:
data_transform = {
    'train': transforms.Compose([
        transforms.Resize(size=(224, 224)),
        transforms.CenterCrop(size=180),
        transforms.ColorJitter(brightness=0.1, contrast=0.1,
                               saturation=0.1),
        transforms.RandomHorizontalFlip(),
        transforms.GaussianBlur(3),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406),
                             (0.229, 0.224, 0.225)),
    ]),
    'val': transforms.Compose([
        transforms.Resize(size=(224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406),
                             (0.229, 0.224, 0.225)),
    ])
}


## 5. Create PyTorch dataset

In [ ]:
class VQADataset(Dataset):
    def __init__(
        self,
        data,
        label2idx,
        max_seq_len=20,
        transform=None,
        img_dir=IMAGE_DIR
    ):
        self.data = data
        self.label2idx = label2idx
        self.max_seq_len = max_seq_len
        self.transform = transform
        self.img_dir = img_dir

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        sample = self.data[index]
        image = Image.open(os.path.join(self.img_dir, sample['image_path'])).convert('RGB')
        if self.transform:
            image = self.transform(image)
        question = tokenize(sample['question'], self.max_seq_len)
        question = torch.tensor(question, dtype=torch.long)
        return image, question, self.label2idx[sample['answer']]


In [ ]:
train_dataset = VQADataset(
    train_data,
    label2idx=label2idx,
    transform=data_transform['train']
)
val_dataset = VQADataset(
    val_data,
    label2idx=label2idx,
    transform=data_transform['val']
)
test_dataset = VQADataset(
    test_data,
    label2idx=label2idx,
    transform=data_transform['val']
)

train_batch_size = 256
test_batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=train_batch_size,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=test_batch_size,
    shuffle=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=test_batch_size,
    shuffle=False
)

images, questions, labels = next(iter(train_loader))
print('Images:', images.shape)
print('Questions:', questions.shape)
print('Labels:', labels.shape)


## 6. Create VQA model

In [ ]:
class VQAModel(nn.Module):
    def __init__(self, n_classes, img_model_name, embedding_dim,
                 n_layers=2, hidden_size=256, drop_p=0.2):
        super().__init__()
        self.image_encoder = timm.create_model(
            img_model_name, pretrained=True, num_classes=hidden_size)
        self.embedding = nn.Embedding(len(vocab), embedding_dim)
        self.lstm1 = nn.LSTM(
            input_size=embedding_dim, hidden_size=hidden_size,
            num_layers=n_layers, batch_first=True,
            bidirectional=True, dropout=drop_p if n_layers > 1 else 0)
        self.fc1 = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size), nn.GELU(),
            nn.Dropout(drop_p), nn.Linear(hidden_size, n_classes))

    def forward(self, image, question):
        image_features = self.image_encoder(image)
        text_features, _ = self.lstm1(self.embedding(question))
        text_features = text_features[:, -1, :]
        return self.fc1(torch.cat((text_features, image_features), dim=1))


In [ ]:
n_classes = len(classes)
img_model_name = 'resnet18'
hidden_size = 256
n_layers = 2
embedding_dim = 128
drop_p = 0.2

model = VQAModel(
    n_classes=n_classes,
    img_model_name=img_model_name,
    embedding_dim=embedding_dim,
    n_layers=n_layers,
    hidden_size=hidden_size,
    drop_p=drop_p
).to(device)

model.eval()
with torch.no_grad():
    output = model(images.to(device), questions.to(device))
print('Output shape:', output.shape)


## 7. Training

In [ ]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for images, questions, labels in dataloader:
            images, questions, labels = (images.to(device), questions.to(device),
                                          labels.to(device))
            outputs = model(images, questions)
            total_loss += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

def fit(model, train_loader, val_loader, criterion, optimizer,
        scheduler, device, epochs):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = -1.0
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, questions, labels in train_loader:
            images, questions, labels = (images.to(device), questions.to(device),
                                          labels.to(device))
            optimizer.zero_grad(set_to_none=True)
            outputs = model(images, questions)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss, train_acc = total_loss / total, correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        for key, value in zip(history, [train_loss, val_loss, train_acc, val_acc]):
            history[key].append(value)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({'model_state_dict': model.state_dict(),
                        'vocab': vocab, 'label2idx': label2idx,
                        'epoch': epoch + 1, 'val_acc': val_acc,
                        'config': config}, CHECKPOINT_DIR / 'best.pt')
        (CHECKPOINT_DIR / 'history.json').write_text(json.dumps(history, indent=2))
        print(f'EPOCH {epoch + 1}: Train loss: {train_loss:.4f} '
              f'Train Acc: {train_acc:.4f} Val loss: {val_loss:.4f} '
              f'Val Acc: {val_acc:.4f}')
        scheduler.step()
    return history


In [ ]:
lr = 1e-3
epochs = 50
config = {'seed': seed, 'img_model_name': img_model_name,
          'hidden_size': hidden_size, 'n_layers': n_layers,
          'embedding_dim': embedding_dim, 'drop_p': drop_p,
          'max_seq_len': 20, 'lr': lr, 'epochs': epochs,
          'train_batch_size': train_batch_size, 'eval_batch_size': test_batch_size}
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=max(1, int(epochs * 0.8)), gamma=0.1)


In [ ]:
history = fit(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs
)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_range = range(1, len(history['train_loss']) + 1)

ax[0].plot(epochs_range, history['train_loss'], label='Train')
ax[0].plot(epochs_range, history['val_loss'], label='Val', color='tab:orange')
ax[0].set_title('Loss')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Cross-entropy loss')
ax[0].legend()

ax[1].plot(epochs_range, history['train_acc'], label='Train')
ax[1].plot(epochs_range, history['val_acc'], label='Val', color='tab:orange')
ax[1].set_title('Accuracy')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
ax[1].legend()
plt.tight_layout()
plt.show()


## 8. Evaluate the best validation checkpoint

Select the model using validation accuracy only. The test split is held out until
this evaluation. Checkpoints include vocabulary, label mapping and configuration.


In [ ]:
checkpoint = torch.load(CHECKPOINT_DIR / 'best.pt', map_location=device, weights_only=True)
if checkpoint['vocab'] != vocab or checkpoint['label2idx'] != label2idx:
    raise ValueError('Checkpoint vocabulary or labels do not match this session.')
model.load_state_dict(checkpoint['model_state_dict'])
val_loss, val_acc = evaluate(model, val_loader, criterion, device)
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
metrics = {'checkpoint_epoch': checkpoint['epoch'], 'val_accuracy': val_acc,
           'test_accuracy': test_acc, 'val_loss': val_loss, 'test_loss': test_loss}
(CHECKPOINT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))


## 9. Results analysis

One accuracy number does not tell us *what* the model learned. We look at the learning curves (above), the accuracy per question type, and a few predictions.

In [ ]:
def predict(model, dataloader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for image, question, labels in dataloader:
            outputs = model(image.to(device), question.to(device))
            y_pred += outputs.argmax(1).cpu().tolist()
            y_true += labels.tolist()
    return np.array(y_true), np.array(y_pred)

y_true, y_pred = predict(model, test_loader, device)
print(f'Test accuracy: {(y_true == y_pred).mean():.4f}')


In [ ]:
# Group the questions by their first two words ('Is this', 'Are there', ...)
def question_type(question):
    return ' '.join(question.split()[:2]).capitalize()

print('Train question types:', Counter(question_type(s['question']) for s in train_data))
first_words = np.array([question_type(s['question']) for s in test_data])
top_types = [w for w, _ in Counter(first_words).most_common(6)]

model_acc = [(y_pred[first_words == w] == y_true[first_words == w]).mean() for w in top_types]
counts = [(first_words == w).sum() for w in top_types]

x = np.arange(len(top_types))
plt.figure(figsize=(9, 4))
bars = plt.bar(x, model_acc, width=0.5, color='tab:blue')
plt.bar_label(bars, labels=[f'{a:.1%}' for a in model_acc])
plt.xticks(x, [f'{w}\n(n={c})' for w, c in zip(top_types, counts)])
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.title('CNN+LSTM: test accuracy by question type (first two words)')
plt.show()


In [ ]:
# Random test samples: green = correct, red = wrong
rng = np.random.default_rng(seed)
ids = rng.choice(len(test_data), size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.flat, ids):
    sample = test_data[i]
    pred = idx2label[y_pred[i]]
    color = 'green' if pred == sample['answer'] else 'red'
    ax.imshow(Image.open(os.path.join(IMAGE_DIR, sample['image_path'])))
    ax.set_title(f"{sample['question']}\nLabel: {sample['answer']} | Pred: {pred}",
                 fontsize=9, color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()


### Interpreting the results

Compare training and validation curves before drawing conclusions. A rising
validation loss with improving training accuracy suggests overfitting.

This is a concatenation baseline, with no cross-modal attention. The text encoder
uses the last padded position, matching the original model; length-aware pooling
or packed sequences are useful follow-up experiments. Spatial augmentation may
change the answer to location-sensitive questions, so inspect examples as well.

The supplied file contains 16 training log entries, with a best recorded validation
accuracy of **63.83% at epoch 13**. It contains no executed test evaluation. The
earlier 65.4% test claim and MLP comparison are therefore excluded from the report.
New results should come from the metrics produced above, after a complete run.


## 10. Inference

In [ ]:
def inference(img_path, question, model, device, max_seq_len=20):
    image = data_transform['val'](Image.open(img_path).convert('RGB'))
    tokens = torch.tensor(tokenize(question, max_seq_len), dtype=torch.long)
    model.eval()
    with torch.inference_mode():
        output = model(image.unsqueeze(0).to(device), tokens.unsqueeze(0).to(device))
    return idx2label[output.argmax(1).item()]


In [ ]:
sample = test_data[0]
img_path = os.path.join(IMAGE_DIR, sample['image_path'])
question = sample['question']
label = sample['answer']
prediction = inference(img_path, question, model, device)

plt.imshow(Image.open(img_path).convert('RGB'))
plt.xticks([])
plt.yticks([])
plt.xlabel(
    f'Question: {question}\nLabel: {label}\nPrediction: {prediction}',
    loc='left',
    family='monospace'
)
plt.show()
